# Lesson 9.1: Study Design: How the Data Were Collected Decides What You Can Claim

Simulate 200 people with a hidden fitness level and see that a coin-flip assignment balances it between two groups while self-selection does not, which biases the estimated effect.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A city health office wants to know whether its new fitness programme really improves people's health scores. It could let residents sign up if they like, or it could flip a coin to decide who gets in. To see which design can be trusted, we invent 200 people, give the programme a true effect of +5 points that we know, and check which design gets close to 5.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Create people with a hidden trait

We invent 200 people. Each has a fitness score that we, as researchers, would not normally see. The outcome (say, a health score) depends on fitness and on a program that truly adds 5 points. A seed makes the simulation repeatable.

In [1]:
import numpy as np  # Load NumPy, the library for numbers and random draws.

rng = np.random.default_rng(42)  # Create a random number generator with a fixed seed (42), so every run gives the same 'random' people and the lab is repeatable.
n = 200  # The number of people in our invented study.
true_effect = 5  # The programme's true effect, which we set ourselves at +5 points. In real life nobody knows this.

fitness = rng.normal(50, 10, n)  # Give each person a hidden fitness level, drawn from a normal curve with mean 50 and SD 10.
noise = rng.normal(0, 5, n)  # Draw random noise (SD 5) for everything else that affects health, such as luck and genes.
base_outcome = 40 + 0.5 * fitness + noise  # Health score without the programme: it depends on fitness (0.5 points per fitness point) plus noise.

print(f'people = {n}, mean fitness = {fitness.mean():.1f}')  # Print the number of people and their mean fitness as a check that the simulation worked.

people = 200, mean fitness = 49.7


**What this means.** We made 200 people with a mean fitness of 49.7. Fitness is hidden in real studies, but here we know it. It raises the health score, so any group that is fitter at the start will also look healthier, whether or not the programme works.

## Step 2: Random assignment: a coin decides

Each person is put in the program or the comparison group by a coin flip, so fitness plays no part. With enough people, the two groups come out alike on fitness, and on every other trait we did not even measure.

In [2]:
random_group = rng.random(n) < 0.5  # Flip a fair coin for every person: draw a random number between 0 and 1, and put the person in the programme if it is below 0.5.

print('RANDOM ASSIGNMENT')  # Print a heading so the output is labelled.
print(f'  program    : n = {random_group.sum()}, mean fitness = {fitness[random_group].mean():.1f}')  # Show the programme group's size and its mean fitness. ~ is the 'not' operator and means the comparison group on the next line.
print(f'  comparison : n = {(~random_group).sum()}, mean fitness = {fitness[~random_group].mean():.1f}')  # Show the comparison group's size and mean fitness. If the coin worked, the two means should be close.

RANDOM ASSIGNMENT
  program    : n = 95, mean fitness = 49.0
  comparison : n = 105, mean fitness = 50.3


**What this means.** The coin put 95 people in the programme and 105 in the comparison group. Their mean fitness is 49.0 and 50.3, only 1.3 apart, so the groups are alike. Randomisation also balances traits we did not measure, which is its great strength.

## Step 3: Self-selection: people choose

Now fitter people are more likely to sign up (a logistic curve turns fitness into a chance of choosing the program). The groups now differ on fitness before the program has done anything.

In [3]:
chance = 1 / (1 + np.exp(-(fitness - 50) / 5))  # Make the chance of signing up rise with fitness. The logistic curve turns fitness into a probability between 0 and 1.
chose = rng.random(n) < chance  # Let each person choose: they join if a random number falls below their own chance. Fitter people join more often.

print('SELF-SELECTION')  # Print a heading so the output is labelled.
print(f'  program    : n = {chose.sum()}, mean fitness = {fitness[chose].mean():.1f}')  # Show the self-selected programme group's size and mean fitness.
print(f'  comparison : n = {(~chose).sum()}, mean fitness = {fitness[~chose].mean():.1f}')  # Show the comparison group's size and mean fitness. Now the groups should differ before the programme has done anything.

SELF-SELECTION
  program    : n = 98, mean fitness = 54.6
  comparison : n = 102, mean fitness = 45.0


**What this means.** When people choose, 98 joined and 102 did not, but the programme group is much fitter: 54.6 against 45.0, a gap of 9.6 points. The groups were different before the programme started, which is confounding.

## Step 4: Compare the estimated effects

The program adds exactly 5 points to whoever gets it. Randomised groups should estimate about 5. In the self-selected groups, the program group was fitter to begin with, so the simple difference over-claims.

In [4]:
for label, group in [('random', random_group), ('self-selected', chose)]:  # Loop over both designs, each as a name and a True/False list saying who was in the programme.
    outcome = base_outcome + true_effect * group  # Each person's outcome is their base score plus 5 points if they were in the programme (True counts as 1).
    diff = outcome[group].mean() - outcome[~group].mean()  # The estimated effect is the programme group's mean outcome minus the comparison group's. This is what a simple study would report.
    print(f'{label:14s} estimated effect = {diff:5.1f}   (true effect = {true_effect})')  # Print the estimate beside the true effect of 5, so we can see how close each design gets.

random         estimated effect =   4.5   (true effect = 5)
self-selected  estimated effect =  10.7   (true effect = 5)


**What this means.** The programme truly adds 5 points. Random assignment estimates 4.5, close to the truth (the gap from 5 is just chance). Self-selection estimates 10.7, more than double, because the fitter people who joined would have scored higher anyway. The design, not the maths, caused the bias.

## So, what is the answer?

Random assignment gave an estimate of 4.5, close to the true +5, because a coin flip balanced fitness between the groups (49.0 against 50.3). Letting people choose gave 10.7, more than double, because fitter people joined and would have scored higher anyway. Caution: this is a simulation with one hidden trait, and random assignment works best with many people (here 200), not guaranteed with few.

## Now you try

A school offers a free tutoring programme that truly adds 3 points to a test score. Students may join if they wish, and the more motivated join more often. Simulate 400 students and see how well a coin flip and self-selection recover the true +3.

**Data:** No data to paste. Simulate: rng = np.random.default_rng(7); n = 400; motivation ~ Normal(50, 10); base score = 60 + 0.6 x motivation + Normal(0, 6); chance of joining = 1 / (1 + exp(-(motivation - 50) / 4)).

1. Simulate the students and assign them by a coin flip (probability 0.5).
2. Let students choose, using the chance of joining above.
3. For each design, find the gap in mean motivation and the estimated programme effect (true effect +3).
4. Which design can you trust, and why?

*Hint:* Copy the lab's structure: draw the trait, build the base outcome, create two True/False group lists, then compare group means for the trait and for outcome = base + 3 x group.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n = 400
motivation = rng.normal(50, 10, n)
base = 60 + 0.6 * motivation + rng.normal(0, 6, n)
coin = rng.random(n) < 0.5
chose = rng.random(n) < 1 / (1 + np.exp(-(motivation - 50) / 4))
for label, g in [('coin flip', coin), ('self-chosen', chose)]:
    y = base + 3 * g
    gap = motivation[g].mean() - motivation[~g].mean()
    print(f'{label:12s} motivation gap = {gap:5.1f}   estimated effect = {y[g].mean() - y[~g].mean():5.1f}')

coin flip    motivation gap =  -0.6   estimated effect =   2.8
self-chosen  motivation gap =  11.9   estimated effect =  10.2


**Answer.** With the coin flip the groups differ by only -0.6 in motivation and the estimated effect is 2.8, close to the true +3. With self-selection the programme group is 11.9 points more motivated and the estimate is 10.2, more than three times the truth. Only the randomised design can be trusted to estimate the effect, because it balances motivation (and every unmeasured trait) between the groups.

## The same question in R, SPSS and Stata

**R**

```r
set.seed(42)  # Fix the random number generator so every run gives the same numbers.
n <- 200  # The number of people in our invented study.
fitness <- rnorm(n, 50, 10)  # Give each person a hidden fitness level from a normal curve with mean 50 and SD 10.
base <- 40 + 0.5 * fitness + rnorm(n, 0, 5)  # Health score without the programme: 40 plus 0.5 x fitness plus random noise with SD 5.
random <- runif(n) < 0.5  # Coin flip for each person: runif gives a random number from 0 to 1, and TRUE (in the programme) means it fell below 0.5.
chose <- runif(n) < plogis((fitness - 50) / 5)  # Self-selection: plogis turns fitness into a chance between 0 and 1, and fitter people are more likely to join.
tapply(fitness, random, mean)  # Mean fitness in each group made by the coin. FALSE is the comparison group and TRUE the programme group.
tapply(fitness, chose, mean)  # The same group means for the self-selected groups. This time the two means should differ clearly.
mean((base + 5 * random)[random]) - mean((base + 5 * random)[!random])  # Estimated effect with random groups: outcome is base + 5 for those in the programme, then programme mean minus comparison mean.
mean((base + 5 * chose)[chose]) - mean((base + 5 * chose)[!chose])  # The same estimate for the self-selected groups, to compare with the true effect of 5.
```

How to read it: The first tapply shows nearly equal mean fitness in the FALSE and TRUE groups (a point or two apart), and the first estimate is near 5. The second tapply shows clearly higher fitness in the TRUE group (about 8 to 10 points), and the second estimate is far above 5 (around 10). Exact numbers differ from Python.

**SPSS** (Transform > Compute Variable (with Random Number functions), then Analyze > Compare Means > Means)

```
SET SEED=42.
INPUT PROGRAM.
LOOP #i = 1 TO 200.
END CASE.
END LOOP.
END FILE.
END INPUT PROGRAM.
COMPUTE fitness = RV.NORMAL(50,10).
COMPUTE random = (RV.UNIFORM(0,1) < 0.5).
COMPUTE chose = (RV.UNIFORM(0,1) < 1/(1+EXP(-(fitness-50)/5))).
MEANS TABLES=fitness BY random /CELLS=MEAN COUNT.
MEANS TABLES=fitness BY chose /CELLS=MEAN COUNT.
```

How to read it: In the first MEANS table the two groups have nearly equal mean fitness. In the second the group with chose = 1 (joined) has clearly higher mean fitness than chose = 0. Exact numbers differ from Python.

**Stata**

```
clear
set seed 42
set obs 200
generate fitness = rnormal(50, 10)
generate base = 40 + 0.5 * fitness + rnormal(0, 5)
generate random = runiform() < 0.5
generate chose = runiform() < invlogit((fitness - 50) / 5)
tabstat fitness, by(random) statistics(mean n)
tabstat fitness, by(chose) statistics(mean n)
generate outcome_r = base + 5 * random
generate outcome_c = base + 5 * chose
ttest outcome_r, by(random)
ttest outcome_c, by(chose)
```

How to read it: The first tabstat shows nearly equal mean fitness; the second shows clearly higher fitness for chose = 1. In the ttest tables, 'diff' is mean(0) - mean(1), so the programme effect appears as a negative number: about -5 for random groups and a larger negative number (around -10) for self-selected groups.